In [1]:

# Assessment 3 - Export all figures and tables
# Run from assessment3/notebooks

from pathlib import Path
import nbformat
import base64
import zipfile
import re
from IPython.display import display

# Folder locations
project = Path.cwd().parent
notebooks_dir = project / "notebooks"
outputs_dir = project / "outputs"

outputs_dir.mkdir(parents=True, exist_ok=True)

# Find all four notebooks (exclude this export notebook)
notebooks = [
    p for p in notebooks_dir.glob("*.ipynb")
    if p.name != "export_all_outputs.ipynb"
    and not p.name.startswith(".")
]

print("Notebooks found:")
for notebook in notebooks:
    print("-", notebook.name)

total_figures = 0
total_tables = 0

for notebook_path in notebooks:

    # Create separate output folder for each notebook
    folder_name = re.sub(
        r"[^a-zA-Z0-9_-]+", "_",
        notebook_path.stem
    )
    save_folder = outputs_dir / folder_name
    save_folder.mkdir(parents=True, exist_ok=True)

    notebook = nbformat.read(
        notebook_path,
        as_version=4
    )

    figure_number = 0
    table_number = 0

    for cell in notebook.cells:

        if cell.cell_type != "code":
            continue

        for output in cell.get("outputs", []):

            data = output.get("data", {})

            # Export figures
            if "image/png" in data:
                figure_number += 1

                image_data = data["image/png"]
                image_path = (
                    save_folder /
                    f"figure_{figure_number:02d}.png"
                )

                image_path.write_bytes(
                    base64.b64decode(image_data)
                )

            # Export displayed pandas tables
            if "text/html" in data:

                html = data["text/html"]

                if "<table" in html.lower():
                    table_number += 1

                    table_path = (
                        save_folder /
                        f"table_{table_number:02d}.html"
                    )

                    table_path.write_text(
                        html,
                        encoding="utf-8"
                    )

    total_figures += figure_number
    total_tables += table_number

    print(
        f"{notebook_path.name}: "
        f"{figure_number} figures, "
        f"{table_number} tables"
    )

# Create one ZIP file with all exported outputs
zip_path = outputs_dir / "all_outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATEDS
) as zip_file:

    for file_path in outputs_dir.rglob("*"):

        if file_path.is_file() and file_path != zip_path:

            zip_file.write(
                file_path,
                arcname=file_path.relative_to(outputs_dir)
            )

print("\nExport completed!")
print("Total figures:", total_figures)
print("Total tables:", total_tables)
print("ZIP saved at:", zip_path)


Notebooks found:
- regression_2_1_data_diwash.ipynb.ipynb
- regression_2_1_model_kshitiz.ipynb
- regression_2_2_data_laxmi.ipynb
- regression_2_2_model_sarika.ipynb
regression_2_1_data_diwash.ipynb.ipynb: 4 figures, 11 tables
regression_2_1_model_kshitiz.ipynb: 4 figures, 3 tables
regression_2_2_data_laxmi.ipynb: 4 figures, 5 tables
regression_2_2_model_sarika.ipynb: 4 figures, 3 tables

Export completed!
Total figures: 16
Total tables: 22
ZIP saved at: C:\Users\kshet\OneDrive\Desktop\FIFA-World-Cup-2026-Project\assessment3\outputs\all_outputs.zip


In [2]:

import nbformat
from pathlib import Path

notebooks_dir = Path.cwd()

for file in notebooks_dir.glob("*.ipynb"):
    if file.name == "export_all_outputs.ipynb":
        continue

    nb = nbformat.read(file, as_version=4)

    figures = 0
    tables = 0
    code_cells = 0

    for cell in nb.cells:
        if cell.cell_type != "code":
            continue

        code_cells += 1

        for output in cell.get("outputs", []):
            data = output.get("data", {})

            if "image/png" in data:
                figures += 1

            if "<table" in data.get("text/html", "").lower():
                tables += 1

    print("\nNotebook:", file.name)
    print("Code cells:", code_cells)
    print("Saved figures:", figures)
    print("Saved tables:", tables)



Notebook: regression_2_1_data_diwash.ipynb.ipynb
Code cells: 16
Saved figures: 4
Saved tables: 11

Notebook: regression_2_1_model_kshitiz.ipynb
Code cells: 8
Saved figures: 4
Saved tables: 3

Notebook: regression_2_2_data_laxmi.ipynb
Code cells: 8
Saved figures: 4
Saved tables: 5

Notebook: regression_2_2_model_sarika.ipynb
Code cells: 9
Saved figures: 4
Saved tables: 3


In [3]:

from pathlib import Path
import pandas as pd

output_dir = Path("../outputs")

records = []

for file in sorted(output_dir.rglob("*")):
    if file.is_file() and file.suffix.lower() in [".png", ".html"]:
        records.append({
            "Notebook": file.parent.name,
            "Type": "Figure" if file.suffix.lower() == ".png" else "Table",
            "Filename": file.name,
            "Path": str(file)
        })

inventory = pd.DataFrame(records)

inventory.to_csv(output_dir / "output_inventory.csv", index=False)

print("Total exported outputs:", len(inventory))
display(inventory.groupby(["Notebook", "Type"]).size())


Total exported outputs: 43


Notebook                          Type  
.ipynb_checkpoints                Figure     4
                                  Table      1
regression_2_1_data_diwash_ipynb  Figure     4
                                  Table     11
regression_2_1_model_kshitiz      Figure     4
                                  Table      3
regression_2_2_data_laxmi         Figure     4
                                  Table      5
regression_2_2_model_sarika       Figure     4
                                  Table      3
dtype: int64

In [ ]:
S